# High-Recall X-Ray Baggage Training (Colab)

Use this notebook on a GPU runtime. It verifies the label mapping first, then trains a stronger YOLO model and exports `best.pt` + `best.onnx` for the web app.

Expected class order:

| id | English | Arabic |
|---:|---|---|
| 0 | gun | ???? |
| 1 | knife | ???? |
| 2 | pliers | ????? |
| 3 | scissors | ??? |
| 4 | wrench | ????? ??? |


In [ ]:
!nvidia-smi
!pip install -q "ultralytics>=8.3.55" onnx onnxslim onnxruntime pillow


## 1. Get the project files

Option A: upload a zip of this repo to Colab and unzip it.
Option B: mount Google Drive if your repo is already there.

Make sure `data/raw/xray-baggage.zip` exists before continuing.


In [ ]:
from pathlib import Path
import os

# If you uploaded xRay_Detection.zip to /content, uncomment:
# !unzip -q /content/xRay_Detection.zip -d /content
# %cd /content/xRay_Detection

# If the repo is in Drive, uncomment and adjust:
# from google.colab import drive
# drive.mount('/content/drive')
# %cd /content/drive/MyDrive/xRay_Detection

ROOT = Path.cwd()
print('Current folder:', ROOT)
assert (ROOT / 'data/raw/xray-baggage.zip').exists(), 'Missing data/raw/xray-baggage.zip'
assert (ROOT / 'scripts/train_high_recall_yolo.py').exists(), 'Run from the repo root'


## 2. Verify Label Mapping Before Training

Open the generated grid and visually confirm that class `1` examples are knives, class `2` are pliers, etc.


In [ ]:
!python scripts/verify_label_mapping.py --zip data/raw/xray-baggage.zip --split train --per-class 5
from IPython.display import Image, display
img = 'runs/audit/label_mapping/train_label_mapping_grid.jpg'
display(Image(filename=img))


## 3. Train

Start with `yolo11m.pt`. For the client version, if time allows on a strong GPU, run the second command with `yolo11l.pt` or `yolo11x.pt`.


In [ ]:
!python scripts/train_high_recall_yolo.py   --device 0   --model yolo11m.pt   --epochs 180   --imgsz 832   --batch 16   --name yolo11m_xray_high_recall   --export


## 4. Evaluate the Exported ONNX on Test Split

This checks that the exported `best.onnx` still has the correct class names and reports per-class AP.


In [ ]:
!python scripts/evaluate_yolo_zip.py   --model data/weights/best.onnx   --zip data/raw/xray-baggage.zip   --split test   --imgsz 832   --batch 16

import json
from pathlib import Path
summary = sorted(Path('runs/evaluate').glob('*/summary.json'))[-1]
print(summary)
print(summary.read_text())


## 5. Refresh Website Demo Images

This regenerates the quick examples shown on the homepage using the newly trained model.


In [ ]:
!python scripts/create_demo_samples.py   --model data/weights/best.onnx   --zip data/raw/xray-baggage.zip   --out services/web/public/demo   --imgsz 832   --min-confidence 0.55   --scan-per-class 1200

!cat services/web/public/demo/manifest.json


## 6. Download Artifacts

Download this zip and copy the files back into the local project, then run `docker compose up -d --build`.


In [ ]:
!mkdir -p artifacts_for_app
!cp data/weights/best.pt artifacts_for_app/best.pt
!cp data/weights/best.onnx artifacts_for_app/best.onnx
!cp -r services/web/public/demo artifacts_for_app/demo
!cp -r runs/evaluate artifacts_for_app/evaluate
!cp -r runs/audit artifacts_for_app/audit
!zip -qr xray_model_artifacts_for_app.zip artifacts_for_app
print('Download: xray_model_artifacts_for_app.zip')
